I am trying to extract all errors LLM generated, initially i was excluding specific error types such as MOJOException etc.

In [8]:
import json
import csv
import re
from pathlib import Path
from typing import Dict, List


# ─── CONFIG ───────────────────────────────────────────────────────────────────

# CSV file with BUMP ground-truth breaking change metadata (shared across all variants)
BUMP_CSV = "/Volumes/Rachna-HD/RQResultsForPaper/RQ2/BUMPErrorLogs/RQ4_resultsBUMP.csv"

# Output CSV — always created as a NEW file with all 9 variants
OUTPUT_CSV = "/Volumes/Rachna-HD/RQResultsForPaper/RQ3/detected_bc_errortype_coverage_v2.csv"

# All 9 (model, context_variant) configurations
VARIANTS = [
    # GPT-4o
    {
        'model': 'GPT-4o',
        'context_variant': 'Minimal',
        'llm_results_json': '/Volumes/Rachna-HD/GPTResults/Exp3BatchResults/bre/transplant_results_breaking_single_module.json',
        'llm_logs_dir': '/Volumes/Rachna-HD/GPTResults/Exp3BatchResults/bre/logs',
    },
    {
        'model': 'GPT-4o',
        'context_variant': 'Method',
        'llm_results_json': '/Volumes/Rachna-HD/GPTResults/Exp6BatchResults/bre/transplant_results_breaking_single_module.json',
        'llm_logs_dir': '/Volumes/Rachna-HD/GPTResults/Exp6BatchResults/bre/logs',
    },
    {
        'model': 'GPT-4o',
        'context_variant': 'Class',
        'llm_results_json': '/Volumes/Rachna-HD/GPTResults/Exp7BatchResultsOp2/bre/transplant_results_breaking_single_module.json',
        'llm_logs_dir': '/Volumes/Rachna-HD/GPTResults/Exp7BatchResultsOp2/bre/logs',
    },
    # Qwen-480B
    {
        'model': 'Qwen-480B',
        'context_variant': 'Minimal',
        'llm_results_json': '/Volumes/Rachna-HD/Qwen480Results/Exp3BatchResults/bre/transplant_results_breaking_single_module.json',
        'llm_logs_dir': '/Volumes/Rachna-HD/Qwen480Results/Exp3BatchResults/bre/logs',
    },
    {
        'model': 'Qwen-480B',
        'context_variant': 'Method',
        'llm_results_json': '/Volumes/Rachna-HD/Qwen480Results/Exp6BatchResults/bre/transplant_results_breaking_single_module.json',
        'llm_logs_dir': '/Volumes/Rachna-HD/Qwen480Results/Exp6BatchResults/bre/logs',
    },
    {
        'model': 'Qwen-480B',
        'context_variant': 'Class',
        'llm_results_json': '/Volumes/Rachna-HD/Qwen480Results/Exp7BatchResults/bre/transplant_results_breaking_single_module.json',
        'llm_logs_dir': '/Volumes/Rachna-HD/Qwen480Results/Exp7BatchResults/bre/logs',
    },
    # GPT-OSS-120b
    {
        'model': 'GPT-OSS-120b',
        'context_variant': 'Minimal',
        'llm_results_json': '/Volumes/Rachna-HD/GPTOSSResults/Exp3BatchResults/bre/transplant_results_breaking_single_module.json',
        'llm_logs_dir': '/Volumes/Rachna-HD/GPTOSSResults/Exp3BatchResults/bre/logs',
    },
    {
        'model': 'GPT-OSS-120b',
        'context_variant': 'Method',
        'llm_results_json': '/Volumes/Rachna-HD/GPTOSSResults/Exp6BatchResults/bre/transplant_results_breaking_single_module.json',
        'llm_logs_dir': '/Volumes/Rachna-HD/GPTOSSResults/Exp6BatchResults/bre/logs',
    },
    {
        'model': 'GPT-OSS-120b',
        'context_variant': 'Class',
        'llm_results_json': '/Volumes/Rachna-HD/GPTOSSResults/Exp7BatchResults/bre/transplant_results_breaking_single_module.json',
        'llm_logs_dir': '/Volumes/Rachna-HD/GPTOSSResults/Exp7BatchResults/bre/logs',
    },
]

# ──────────────────────────────────────────────────────────────────────────────


class LogParser:
    """
    Parse Maven/Java test logs and extract all error information.
    Same logic as Script 2 (bump_executor.py) LogParser.
    """

    def __init__(self, log_text: str):
        self.log = log_text

    def parse(self) -> Dict:
        """Parse log and extract all error information."""
        return {
            'all_exceptions': self.extract_exceptions(),
            'all_errors': self.extract_errors(),
            'failed_tests': self.extract_failed_tests(),
            'compilation_failures': self.extract_compilation_failures(),
            'error_messages': self.extract_error_messages(),
            'stack_traces': self.extract_stack_traces(),
            'maven_errors': self.extract_maven_errors(),
            'test_summary': self.extract_test_summary(),
        }

    def extract_exceptions(self) -> List[Dict]:
        """Extract all Java exceptions from the log."""
        exceptions = []

        # Pattern for Java exceptions
        pattern = r'([\w.]+(?:Exception|Error))(?::\s*(.+?))?(?=\n|\r|$)'

        for match in re.finditer(pattern, self.log):
            exception_type = match.group(1)
            exception_message = match.group(2).strip() if match.group(2) else ""

            # Get line number
            line_num = self.log[:match.start()].count('\n') + 1

            # Get context (50 chars before and after)
            start = max(0, match.start() - 50)
            end = min(len(self.log), match.end() + 100)
            context = self.log[start:end].replace('\n', ' ').strip()

            exceptions.append({
                'type': exception_type,
                'message': exception_message[:200],
                'line': line_num,
                'context': context[:200]
            })

        return exceptions

    def extract_errors(self) -> List[str]:
        """Extract all [ERROR] lines from Maven output."""
        errors = []
        pattern = r'\[ERROR\]\s*(.+?)(?=\n|$)'

        for match in re.finditer(pattern, self.log, re.MULTILINE):
            error_text = match.group(1).strip()
            if error_text and len(error_text) > 5:
                errors.append(error_text[:300])

        return list(dict.fromkeys(errors))

    def extract_failed_tests(self) -> List[Dict]:
        """Extract information about failed tests."""
        failed_tests = []

        # Pattern 1: JUnit format - TestName(ClassName)
        pattern1 = r'(\w+)\(([\w.]+)\)\s+Time elapsed:.*?<<<\s*(FAILURE|ERROR)!'
        for match in re.finditer(pattern1, self.log):
            failed_tests.append({
                'test_method': match.group(1),
                'test_class': match.group(2),
                'failure_type': match.group(3),
                'format': 'junit'
            })

        # Pattern 2: Maven format - package.Class.method
        pattern2 = r'Failed tests?:\s+([\w.]+\.[\w.]+)'
        for match in re.finditer(pattern2, self.log):
            full_name = match.group(1)
            parts = full_name.rsplit('.', 1)
            failed_tests.append({
                'test_method': parts[1] if len(parts) > 1 else full_name,
                'test_class': parts[0] if len(parts) > 1 else 'Unknown',
                'failure_type': 'FAILURE',
                'format': 'maven'
            })

        # Pattern 3: Errors in tests
        pattern3 = r'Errors?:\s+([\w.]+\.[\w.]+)'
        for match in re.finditer(pattern3, self.log):
            full_name = match.group(1)
            parts = full_name.rsplit('.', 1)
            failed_tests.append({
                'test_method': parts[1] if len(parts) > 1 else full_name,
                'test_class': parts[0] if len(parts) > 1 else 'Unknown',
                'failure_type': 'ERROR',
                'format': 'maven'
            })

        return failed_tests

    def extract_compilation_failures(self) -> List[Dict]:
        """Extract compilation errors."""
        compilation_errors = []

        if re.search(r'COMPILATION ERROR', self.log, re.IGNORECASE):
            pattern = r'\[ERROR\]\s*([\w/\\.]+\.java):\[(\d+),(\d+)\]\s*(.+?)(?=\n|$)'
            for match in re.finditer(pattern, self.log):
                compilation_errors.append({
                    'file': match.group(1),
                    'line': match.group(2),
                    'column': match.group(3),
                    'error': match.group(4).strip()[:200]
                })

        return compilation_errors

    def extract_error_messages(self) -> List[str]:
        """Extract readable error messages."""
        messages = []

        pattern1 = r'expected:\s*<?(.+?)>?\s*but was:\s*<?(.+?)>?'
        for match in re.finditer(pattern1, self.log, re.IGNORECASE):
            messages.append(f"Expected: {match.group(1)}, but was: {match.group(2)}")

        pattern2 = r'(?:Exception|Error):\s*([^\n]{20,200})'
        for match in re.finditer(pattern2, self.log):
            msg = match.group(1).strip()
            if msg not in str(messages):
                messages.append(msg)

        return messages[:10]

    def extract_stack_traces(self) -> List[str]:
        """Extract stack traces."""
        stack_traces = []

        pattern = r'((?:[\w.]+(?:Exception|Error)[^\n]*(?:\n\s+at\s+[\w.$<>]+\([^\)]+\))+))'

        matches = re.finditer(pattern, self.log, re.MULTILINE)
        for match in matches:
            trace = match.group(1)
            lines = trace.split('\n')[:6]
            stack_traces.append('\n'.join(lines))

        return stack_traces[:3]

    def extract_maven_errors(self) -> Dict:
        """Extract Maven-specific error information."""
        maven_info = {
            'build_failure': bool(re.search(r'BUILD FAILURE', self.log)),
            'build_success': bool(re.search(r'BUILD SUCCESS', self.log)),
            'reactor_summary': None,
            'failure_message': None
        }

        failure_pattern = r'(?:Failure|Error)\s*message:\s*(.+?)(?=\n|$)'
        match = re.search(failure_pattern, self.log, re.IGNORECASE)
        if match:
            maven_info['failure_message'] = match.group(1).strip()

        return maven_info

    def extract_test_summary(self) -> Dict:
        """Extract test execution summary."""
        summary = {
            'tests_run': 0,
            'failures': 0,
            'errors': 0,
            'skipped': 0
        }

        pattern = r'Tests run:\s*(\d+).*?Failures:\s*(\d+).*?Errors:\s*(\d+).*?Skipped:\s*(\d+)'
        match = re.search(pattern, self.log)

        if match:
            summary['tests_run'] = int(match.group(1))
            summary['failures'] = int(match.group(2))
            summary['errors'] = int(match.group(3))
            summary['skipped'] = int(match.group(4))

        return summary


def parse_bump_errors(raw: str) -> set:
    """
    Parse the exception_types column from BUMP CSV into a set of
    short exception class names (e.g. 'NoClassDefFoundError').
    Pipe-separated values are split.
    NO exclusions — all exception/error types are kept.
    """
    if not raw or str(raw).strip() in ('', 'nan'):
        return set()
    result = set()
    for e in str(raw).split('|'):
        e = e.strip()
        if not e:
            continue
        result.add(e.split('.')[-1])
    return result


def parse_log_errors(log_path: Path) -> set:
    """
    Run the full LogParser (same as Script 2) on a test execution log file
    and return all unique short exception/error class names found.
    Uses extract_exceptions() which captures every Exception/Error with
    no exclusions.
    """
    if not log_path.exists():
        return set()
    try:
        content = log_path.read_text(encoding='utf-8', errors='ignore')
    except Exception:
        return set()

    parser = LogParser(content)
    parsed = parser.parse()

    found = set()
    for exc in parsed['all_exceptions']:
        full_name = exc['type']
        short_name = full_name.split('.')[-1]
        if short_name:
            found.add(short_name)

    return found


def load_bump(bump_csv: str) -> dict:
    """
    Load BUMP CSV into a dict keyed by instance ID (custom_id).
    Returns per-instance: bump_errors (set), bump_errors_raw (str), failure_category (str).
    """
    data = {}
    with open(bump_csv, encoding='utf-8') as f:
        for row in csv.DictReader(f):
            data[row['custom_id']] = {
                'bump_errors':      parse_bump_errors(row.get('exception_types', '')),
                'bump_errors_raw':  row.get('exception_types', ''),
                'failure_category': row.get('failureCategory', ''),
            }
    return data


def load_llm(llm_json: str) -> dict:
    """
    Load LLM test execution results JSON.
    Returns the 'results' dict keyed by instance ID.
    """
    with open(llm_json) as f:
        data = json.load(f)
    return data.get('results', data)


def process_variant(variant: dict, bump: dict) -> list:
    """
    Process a single (model, context_variant) configuration.
    Returns a list of row dicts for the output CSV.
    """
    model_name = variant['model']
    context_variant = variant['context_variant']
    llm_results_json = variant['llm_results_json']
    llm_logs_dir = variant['llm_logs_dir']

    print(f"\n{'='*70}")
    print(f"Processing: {model_name} | {context_variant}")
    print(f"  JSON: {llm_results_json}")
    print(f"  Logs: {llm_logs_dir}")
    print(f"{'='*70}")

    llm = load_llm(llm_results_json)
    logs = Path(llm_logs_dir)

    rows = []

    for instance_id, instance_data in llm.items():
        failed_tests = instance_data.get('tests', {}).get('failed', [])

        # Only process instances where LLM detected a BC (has failed tests on v2)
        if not failed_tests:
            continue
        if instance_id not in bump:
            print(f"  Warning: {instance_id} not in BUMP CSV — skipping")
            continue

        bump_errors = bump[instance_id]['bump_errors']

        # Collect all exception types from failed test logs
        # Skip transplant_issue tests, process all remaining test log files
        llm_errors = set()
        for test in failed_tests:
            if test.get('result_type') == 'transplant_issue':
                continue
            test_file = test.get('file', '')
            if not test_file:
                continue
            log_path = logs / f"{instance_id}_{test_file}_breaking_single.log"
            llm_errors.update(parse_log_errors(log_path))

        matched = bump_errors & llm_errors
        missed  = bump_errors - llm_errors
        new     = llm_errors  - bump_errors

        match_rate = round(len(matched) / len(bump_errors) * 100, 1) if bump_errors else 0.0

        rows.append({
            'model':               model_name,
            'context_variant':     context_variant,
            'instance':            instance_id,
            'bump_bc_failure_category': bump[instance_id]['failure_category'],
            'bump_bc_errors_raw':  bump[instance_id]['bump_errors_raw'],
            'bump_bc_errors':      '|'.join(sorted(bump_errors)),
            'llm_detected_errors': '|'.join(sorted(llm_errors)),
            'matched_errors':      '|'.join(sorted(matched)),
            'missed_errors':       '|'.join(sorted(missed)),
            'new_errors':          '|'.join(sorted(new)),
            'bump_total':          len(bump_errors),
            'llm_total':           len(llm_errors),
            'matched_count':       len(matched),
            'missed_count':        len(missed),
            'new_count':           len(new),
            'match_rate_%':        match_rate,
            'num_failed_tests':    len(failed_tests),
        })

    print(f"  Done. {len(rows)} detected instances for {model_name} | {context_variant}")
    return rows


def main():
    # Load BUMP ground truth once (shared across all variants)
    bump = load_bump(BUMP_CSV)
    print(f"Loaded BUMP CSV: {len(bump)} instances")

    all_rows = []

    for variant in VARIANTS:
        rows = process_variant(variant, bump)
        all_rows.extend(rows)

    # Write CSV — always create a NEW file with all variants
    Path(OUTPUT_CSV).parent.mkdir(parents=True, exist_ok=True)
    fieldnames = [
        'model', 'context_variant',
        'instance', 'bump_bc_failure_category', 'bump_bc_errors_raw',
        'bump_bc_errors', 'llm_detected_errors', 'matched_errors', 'missed_errors', 'new_errors',
        'bump_total', 'llm_total', 'matched_count', 'missed_count', 'new_count',
        'match_rate_%', 'num_failed_tests',
    ]
    with open(OUTPUT_CSV, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_rows)

    print(f"\n{'='*70}")
    print(f"ALL DONE. {len(all_rows)} total rows written to {OUTPUT_CSV}")
    print(f"{'='*70}")
    for variant in VARIANTS:
        count = sum(1 for r in all_rows if r['model'] == variant['model'] and r['context_variant'] == variant['context_variant'])
        print(f"  {variant['model']} | {variant['context_variant']}: {count} instances")


if __name__ == "__main__":
    main()

Loaded BUMP CSV: 169 instances

Processing: GPT-4o | Minimal
  JSON: /Volumes/Rachna-HD/GPTResults/Exp3BatchResults/bre/transplant_results_breaking_single_module.json
  Logs: /Volumes/Rachna-HD/GPTResults/Exp3BatchResults/bre/logs
  Done. 17 detected instances for GPT-4o | Minimal

Processing: GPT-4o | Method
  JSON: /Volumes/Rachna-HD/GPTResults/Exp6BatchResults/bre/transplant_results_breaking_single_module.json
  Logs: /Volumes/Rachna-HD/GPTResults/Exp6BatchResults/bre/logs
  Done. 13 detected instances for GPT-4o | Method

Processing: GPT-4o | Class
  JSON: /Volumes/Rachna-HD/GPTResults/Exp7BatchResultsOp2/bre/transplant_results_breaking_single_module.json
  Logs: /Volumes/Rachna-HD/GPTResults/Exp7BatchResultsOp2/bre/logs
  Done. 27 detected instances for GPT-4o | Class

Processing: Qwen-480B | Minimal
  JSON: /Volumes/Rachna-HD/Qwen480Results/Exp3BatchResults/bre/transplant_results_breaking_single_module.json
  Logs: /Volumes/Rachna-HD/Qwen480Results/Exp3BatchResults/bre/logs
  Don

In [9]:
"""
add_error_category_to_bump_v2.py
=================================
Adds normalized_error_category column to BUMP CSV.
Reads exception_types, extracts short class names, keeps ALL error types
with NO exclusions and NO hardcoded mapping.
Each unique exception/error class name becomes its own category.
Writes to a new v2 file to avoid corrupting the original.
"""

import csv
import pandas as pd
from collections import Counter


# ─── CONFIG ───────────────────────────────────────────────────────────────────
BUMP_CSV   = "/Volumes/Rachna-HD/ConfigFiles/Candidate_BUMP_Instance_errorTypes.csv"
OUTPUT_CSV = "/Volumes/Rachna-HD/ConfigFiles/Candidate_BUMP_Instance_errorTypes_with_categories_v2.csv"
# ──────────────────────────────────────────────────────────────────────────────


def categorize_exceptions(raw: str) -> str:
    """
    Extract all exception/error short class names from the pipe-separated
    exception_types column. No exclusions, no hardcoded mapping.
    Each unique short class name is kept as-is.
    Returns pipe-separated string of unique short class names, or empty string
    if none found.
    """
    if not raw or str(raw).strip() in ('', 'nan'):
        return ''

    categories = []
    for e in str(raw).split('|'):
        # Strip package prefix to get short class name
        short = e.strip().split('.')[-1]
        if not short:
            continue
        # Keep every unique short class name — no exclusions
        if short not in categories:
            categories.append(short)

    return '|'.join(categories)


def detect_sep(path: str) -> str:
    with open(path, encoding='utf-8') as f:
        header = f.readline()
    tabs   = header.count('\t')
    commas = header.count(',')
    sep = '\t' if tabs > commas else ','
    print(f"  Separator: {'TAB' if sep == chr(9) else 'COMMA'} (tabs={tabs}, commas={commas})")
    return sep


def main():
    sep = detect_sep(BUMP_CSV)
    df  = pd.read_csv(BUMP_CSV, sep=sep, dtype=str, keep_default_na=False,
                      quoting=csv.QUOTE_MINIMAL)
    print(f"  Loaded {len(df)} rows, {len(df.columns)} columns")
    print(f"  Columns: {list(df.columns)}")

    # Debug: show raw exception_types for first 5 rows
    print(f"\n  Sample exception_types (first 5 rows):")
    for _, row in df.head(5).iterrows():
        raw = row.get('exception_types', '')
        cat = categorize_exceptions(raw)
        print(f"    {row['custom_id']:8s}  raw: {raw[:80]}")
        print(f"    {'':8s}  cat: {cat}")

    # Add normalized column
    df['normalized_error_category'] = df['exception_types'].apply(categorize_exceptions)

    # Summary: count how many instances each unique error type appears in
    cat_counts: Counter = Counter()
    empty_count = 0
    for val in df['normalized_error_category']:
        if not val:
            empty_count += 1
            continue
        for c in str(val).split('|'):
            c = c.strip()
            if c:
                cat_counts[c] += 1

    print(f"\n  All unique error types ({len(cat_counts)} types):")
    for cat, cnt in sorted(cat_counts.items(), key=lambda x: -x[1]):
        print(f"    {cat:50s}  {cnt:3d} instances")
    if empty_count:
        print(f"    {'(no exception_types)':50s}  {empty_count:3d} instances")

    # Write to new file, same separator
    df.to_csv(OUTPUT_CSV, sep=sep, index=False, quoting=csv.QUOTE_MINIMAL)
    print(f"\n  Saved → {OUTPUT_CSV}  ({len(df.columns)} columns, {len(df)} rows)")
    print(f"  Original {BUMP_CSV} is untouched.")


if __name__ == "__main__":
    main()

  Separator: COMMA (tabs=0, commas=30)
  Loaded 89 rows, 31 columns
  Columns: ['custom_id', 'clientGithubURL', 'clientProject', 'clientProjectOrganisation', 'breakingCommit', 'dependencyGroupID', 'dependencyArtifactID', 'previousVersion', 'newVersion', 'failureCategory', 'docker_image_breaking', 'execution_timestamp', 'execution_success', 'return_code', 'execution_time_seconds', 'tests_run', 'test_failures', 'test_errors', 'test_skipped', 'num_exceptions', 'num_failed_tests', 'num_compilation_errors', 'num_error_messages', 'exception_types', 'first_error_message', 'all_maven_errors', 'build_status', 'error_category', 'notes', 'log_file', 'parsed_errors_file']

  Sample exception_types (first 5 rows):
    BBC01     raw: java.lang.NoClassDefFoundError|StopException|java.lang.ClassNotFoundException|Mo
              cat: NoClassDefFoundError|StopException|ClassNotFoundException|MojoFailureException
    BBC02     raw: java.lang.NoClassDefFoundError|java.lang.RuntimeException|java.lang.Clas

In [12]:
"""
analysis1_error_type_coverage_v2.py
====================================
Pipeline:
  1. BUMP CSV      → reads normalized_error_category column
                     all_cats   = ALL unique categories across the full BUMP CSV
                     bump_total = # instances per category from the FULL BUMP CSV
  2. Detected CSV  → per (model, context_variant, instance):
                       bump_bc_errors     : what BUMP ground truth has for this instance
                       llm_detected_errors: what LLM threw for this instance
  3. For each (model, context_variant, error_category):
       bump_total   = # BUMP instances (full CSV) with this category
       llm_detected = # instances where LLM detected this category
       in_common    = # instances where both BUMP and LLM share this category
       rate         = in_common / bump_total * 100

No exclusions — all error types are kept, consistent with v2 pipeline.
"""

import pandas as pd
from collections import defaultdict
from pathlib import Path


# ─── CONFIG ───────────────────────────────────────────────────────────────────
BUMP_CSV         = "/Volumes/Rachna-HD/ConfigFiles/Candidate_BUMP_Instance_errorTypes_with_categories_v2.csv"
DETECTED_CSV     = "/Volumes/Rachna-HD/RQResultsForPaper/RQ3/detected_bc_errortype_coverage_v2.csv"
OUTPUT_TABLE_CSV = "/Volumes/Rachna-HD/RQResultsForPaper/RQ3/analysis1_error_type_coverage_table_v2.csv"

MODEL_ORDER   = ['GPT-4o', 'Qwen-480B', 'GPT-OSS-120b']
VARIANT_ORDER = ['Minimal', 'Method', 'Class']
# ──────────────────────────────────────────────────────────────────────────────


def parse_pipe(val) -> set[str]:
    """Pipe-separated string → set of short class names. No exclusions."""
    if not val or str(val).strip() in ('', 'nan'):
        return set()
    return {e.strip().split('.')[-1] for e in str(val).split('|') if e.strip()}


# ─── STEP 1: Load BUMP CSV ────────────────────────────────────────────────────

def load_bump(bump_csv: str, detected_ids: set[str]) -> tuple[dict[str, set[str]], list[str], dict[str, int]]:
    """
    Reads normalized_error_category from the full BUMP CSV.

    Returns:
      bump_cats         : dict[custom_id → set of categories], restricted to detected_ids
      all_cats          : list of all unique categories, ordered by frequency descending
      bump_total_by_cat : dict[category → count], from ALL rows in BUMP CSV
    """
    df = pd.read_csv(bump_csv, sep=None, engine='python', dtype=str, keep_default_na=False)
    df['custom_id'] = df['custom_id'].astype(str).str.strip()

    bump_cats: dict[str, set[str]] = {}
    all_counts: dict[str, int] = defaultdict(int)
    detected_counts: dict[str, int] = defaultdict(int)

    for _, row in df.iterrows():
        cid  = row['custom_id']
        cats = parse_pipe(row.get('normalized_error_category', ''))
        # bump_total always from full BUMP CSV
        for c in cats:
            all_counts[c] += 1
        # bump_cats restricted to detected_ids (for in_common calculation)
        if cid in detected_ids:
            bump_cats[cid] = cats
            for c in cats:
                detected_counts[c] += 1

    # Order by frequency descending (full BUMP)
    all_cats = sorted(all_counts.keys(), key=lambda c: -all_counts[c])

    print(f"  BUMP CSV total: {len(df)} instances")
    print(f"  All unique categories ({len(all_cats)}):")
    for c in all_cats:
        print(f"    {c:50s}  full={all_counts[c]}  detected={detected_counts.get(c, 0)}")

    return bump_cats, all_cats, dict(all_counts)


# ─── STEP 2: Load detected CSV ────────────────────────────────────────────────

def load_detected(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, sep=None, engine='python', dtype=str, keep_default_na=False)

    df['custom_id'] = df['custom_id'].astype(str).str.strip()

    # Parse bump_bc_errors and llm_detected_errors into sets of short class names
    df['_bump_cats'] = df['bump_bc_errors'].apply(parse_pipe)
    df['_llm_cats']  = df['llm_detected_errors'].apply(parse_pipe)

    print(f"  Detected CSV: {len(df)} rows | "
          f"{df['custom_id'].nunique()} unique instances | "
          f"{df['model'].nunique()} models | "
          f"{df['context_variant'].nunique()} variants")
    return df


# ─── STEP 3: Build coverage table ────────────────────────────────────────────

def build_table(
    bump_total_by_cat: dict[str, int],
    bump_cats: dict[str, set[str]],
    all_cats: list[str],
    df: pd.DataFrame,
) -> pd.DataFrame:
    """
    For each (model, context_variant, error_category):
      bump_total   : from full BUMP CSV (how many BUMP instances have this error type)
      llm_detected : # instances in detected CSV where LLM threw this error type
      in_common    : # instances where BUMP has this error type AND LLM also detected it
      rate         : in_common / bump_total * 100
    """
    records = []
    for model in MODEL_ORDER:
        for variant in VARIANT_ORDER:
            sub = df[(df['model'] == model) & (df['context_variant'] == variant)]

            for cat in all_cats:
                bump_total = bump_total_by_cat.get(cat, 0)

                # llm_detected: count instances where LLM detected this error type
                llm_detected = int(sub['_llm_cats'].apply(lambda s: cat in s).sum())

                # in_common: count instances where BUMP has this error type
                # AND LLM also detected the same error type
                in_common = int(sub.apply(
                    lambda row: cat in bump_cats.get(row['custom_id'], set())
                                and cat in row['_llm_cats'],
                    axis=1
                ).sum())

                rate = round(in_common / bump_total * 100, 1) if bump_total > 0 else 0.0

                records.append({
                    'model':            model,
                    'context_variant':  variant,
                    'error_category':   cat,
                    'bump_total':       bump_total,
                    'llm_detected':     llm_detected,
                    'in_common':        in_common,
                    'detection_rate_%': rate,
                })

    return pd.DataFrame(records)


# ─── STEP 4: LaTeX tables (one per model) ────────────────────────────────────

def print_latex(table: pd.DataFrame, all_cats: list[str]):
    # Only show categories that have at least one BUMP instance
    active_cats = [c for c in all_cats
                   if table[table['error_category'] == c]['bump_total'].iloc[0] > 0]

    for model in MODEL_ORDER:
        model_df = table[table['model'] == model]

        print(f"\n% ── Model: {model} ──────────────────────────────────────────────")
        print(r"\begin{table}[htbp]")
        print(r"\centering")
        print(f"\\caption{{Error Type Coverage for \\textbf{{{model}}}}}")
        print(f"\\label{{tab:coverage_{model.replace('-', '_').replace(' ', '_')}}}")
        print(r"\resizebox{\columnwidth}{!}{%")
        print(r"\begin{tabular}{l r | r r r | r r r | r r r}")
        print(r"\toprule")
        print(r"& & \multicolumn{3}{c|}{\textit{Minimal}} "
              r"& \multicolumn{3}{c|}{\textit{Method}} "
              r"& \multicolumn{3}{c}{\textit{Class}} \\")
        print(r"\cmidrule(lr){3-5}\cmidrule(lr){6-8}\cmidrule(lr){9-11}")
        print(r"\textbf{Error Category} & \textbf{BUMP $n$} "
              r"& LLM & Com & \% "
              r"& LLM & Com & \% "
              r"& LLM & Com & \% \\")
        print(r"\midrule")

        for cat in active_cats:
            bump_n = model_df[model_df['error_category'] == cat]['bump_total'].iloc[0]
            cells = []
            for variant in VARIANT_ORDER:
                r = model_df[
                    (model_df['context_variant'] == variant) &
                    (model_df['error_category'] == cat)
                ]
                llm  = r['llm_detected'].values[0]
                com  = r['in_common'].values[0]
                rate = r['detection_rate_%'].values[0]
                cells.append(f"{llm} & {com} & {rate}")

            # Escape underscores in category names for LaTeX
            cat_latex = cat.replace('_', r'\_')
            print(f"\\texttt{{{cat_latex}}} & {bump_n} & " + " & ".join(cells) + r" \\")

        print(r"\bottomrule")
        print(r"\end{tabular}%")
        print(r"}")
        print(r"\end{table}")


# ─── MAIN ─────────────────────────────────────────────────────────────────────

def main():
    print("\nStep 1: Loading detected CSV...")
    df = load_detected(DETECTED_CSV)

    print("\nStep 2: Loading BUMP CSV (bump_total from full CSV, rest filtered to detected)...")
    detected_ids = set(df['custom_id'].unique())
    bump_cats, all_cats, bump_total_by_cat = load_bump(BUMP_CSV, detected_ids=detected_ids)

    print("\nStep 3: Building coverage table...")
    table = build_table(bump_total_by_cat, bump_cats, all_cats, df)

    Path(OUTPUT_TABLE_CSV).parent.mkdir(parents=True, exist_ok=True)
    table.to_csv(OUTPUT_TABLE_CSV, index=False)
    print(f"  Table saved → {OUTPUT_TABLE_CSV}")

    print("\nSummary (averaged across model-variants):")
    summary = (
        table[table['bump_total'] > 0]
        .groupby('error_category')
        .agg(
            bump_total=('bump_total', 'first'),
            avg_llm_detected=('llm_detected', 'mean'),
            avg_in_common=('in_common', 'mean'),
            avg_rate=('detection_rate_%', 'mean')
        )
        .round(1)
        .sort_values('avg_rate', ascending=False)
    )
    print(summary.to_string())

    print("\nStep 4: Printing LaTeX tables (one per model)...")
    print_latex(table, all_cats)

    print("\nDone.")


if __name__ == "__main__":
    main()


Step 1: Loading detected CSV...
  Detected CSV: 137 rows | 32 unique instances | 3 models | 3 variants

Step 2: Loading BUMP CSV (bump_total from full CSV, rest filtered to detected)...
  BUMP CSV total: 89 instances
  All unique categories (56):
    MojoFailureException                                full=88  detected=32
    NoClassDefFoundError                                full=35  detected=20
    ClassNotFoundException                              full=30  detected=16
    SocketTimeoutException                              full=15  detected=0
    ClassCastException                                  full=10  detected=7
    AssertionError                                      full=10  detected=0
    RuntimeException                                    full=8  detected=3
    ExceptionInInitializerError                         full=8  detected=1
    UnsupportedClassVersionError                        full=8  detected=0
    BeanInstantiationException                          full=8  dete

In [ ]:
"""
analysis1_error_type_coverage_v2.py
====================================
Pipeline:
  1. BUMP CSV      → reads normalized_error_category column
                     all_cats   = ALL unique categories across the full BUMP CSV
                     bump_total = # instances per category from the FULL BUMP CSV
  2. Detected CSV  → per (model, context_variant, instance):
                       bump_bc_errors     : what BUMP ground truth has for this instance
                       llm_detected_errors: what LLM threw for this instance
  3. For each (model, context_variant, error_category):
       bump_total   = # BUMP instances (full CSV) with this category
       llm_detected = # instances where LLM detected this category
       in_common    = # instances where both BUMP and LLM share this category
       rate         = in_common / bump_total * 100

No exclusions — all error types are kept, consistent with v2 pipeline.
"""

import pandas as pd
from collections import defaultdict
from pathlib import Path


# ─── CONFIG ───────────────────────────────────────────────────────────────────
BUMP_CSV         = "/Volumes/Rachna-HD/ConfigFiles/Candidate_BUMP_Instance_errorTypes_with_categories_v2.csv"
DETECTED_CSV     = "/Volumes/Rachna-HD/RQResultsForPaper/RQ3/detected_bc_errortype_coverage_v2.csv"
OUTPUT_TABLE_CSV = "/Volumes/Rachna-HD/RQResultsForPaper/RQ3/ErrorTypeCoverageTableRQ3.csv"

MODEL_ORDER   = ['GPT-4o', 'Qwen-480B', 'GPT-OSS-120b']
VARIANT_ORDER = ['Minimal', 'Method', 'Class']
# ──────────────────────────────────────────────────────────────────────────────


def parse_pipe(val) -> set[str]:
    """Pipe-separated string → set of short class names. No exclusions."""
    if not val or str(val).strip() in ('', 'nan'):
        return set()
    return {e.strip().split('.')[-1] for e in str(val).split('|') if e.strip()}


# ─── STEP 1: Load BUMP CSV ────────────────────────────────────────────────────

def load_bump(bump_csv: str, detected_ids: set[str]) -> tuple[dict[str, set[str]], list[str], dict[str, int]]:
    """
    Reads normalized_error_category from the full BUMP CSV.

    Returns:
      bump_cats         : dict[custom_id → set of categories], restricted to detected_ids
      all_cats          : list of all unique categories, ordered by frequency descending
      bump_total_by_cat : dict[category → count], from ALL rows in BUMP CSV
    """
    df = pd.read_csv(bump_csv, sep=None, engine='python', dtype=str, keep_default_na=False)
    df['custom_id'] = df['custom_id'].astype(str).str.strip()

    bump_cats: dict[str, set[str]] = {}
    all_counts: dict[str, int] = defaultdict(int)
    detected_counts: dict[str, int] = defaultdict(int)

    for _, row in df.iterrows():
        cid  = row['custom_id']
        cats = parse_pipe(row.get('normalized_error_category', ''))
        # bump_total always from full BUMP CSV
        for c in cats:
            all_counts[c] += 1
        # bump_cats restricted to detected_ids (for in_common calculation)
        if cid in detected_ids:
            bump_cats[cid] = cats
            for c in cats:
                detected_counts[c] += 1

    # Order by frequency descending (full BUMP)
    all_cats = sorted(all_counts.keys(), key=lambda c: -all_counts[c])

    print(f"  BUMP CSV total: {len(df)} instances")
    print(f"  All unique categories ({len(all_cats)}):")
    for c in all_cats:
        print(f"    {c:50s}  full={all_counts[c]}  detected={detected_counts.get(c, 0)}")

    return bump_cats, all_cats, dict(all_counts)


# ─── STEP 2: Load detected CSV ────────────────────────────────────────────────

def load_detected(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, sep=None, engine='python', dtype=str, keep_default_na=False)

    df['custom_id'] = df['custom_id'].astype(str).str.strip()

    # Parse bump_bc_errors and llm_detected_errors into sets of short class names
    df['_bump_cats'] = df['bump_bc_errors'].apply(parse_pipe)
    df['_llm_cats']  = df['llm_detected_errors'].apply(parse_pipe)

    print(f"  Detected CSV: {len(df)} rows | "
          f"{df['custom_id'].nunique()} unique instances | "
          f"{df['model'].nunique()} models | "
          f"{df['context_variant'].nunique()} variants")
    return df


# ─── STEP 3: Build coverage table ────────────────────────────────────────────

def build_table(
    bump_total_by_cat: dict[str, int],
    bump_cats: dict[str, set[str]],
    all_cats: list[str],
    df: pd.DataFrame,
) -> pd.DataFrame:
    """
    For each (model, context_variant, error_category):
      bump_total   : from full BUMP CSV (how many BUMP instances have this error type)
      llm_detected : # instances in detected CSV where LLM threw this error type
      in_common    : # instances where BUMP has this error type AND LLM also detected it
      rate         : in_common / bump_total * 100
    """
    records = []
    for model in MODEL_ORDER:
        for variant in VARIANT_ORDER:
            sub = df[(df['model'] == model) & (df['context_variant'] == variant)]

            for cat in all_cats:
                bump_total = bump_total_by_cat.get(cat, 0)

                # llm_detected: count instances where LLM detected this error type
                llm_detected = int(sub['_llm_cats'].apply(lambda s: cat in s).sum())

                # in_common: count instances where BUMP has this error type
                # AND LLM also detected the same error type
                in_common = int(sub.apply(
                    lambda row: cat in bump_cats.get(row['custom_id'], set())
                                and cat in row['_llm_cats'],
                    axis=1
                ).sum())

                rate = round(in_common / bump_total * 100, 1) if bump_total > 0 else 0.0

                records.append({
                    'model':            model,
                    'context_variant':  variant,
                    'error_category':   cat,
                    'bump_total':       bump_total,
                    'llm_detected':     llm_detected,
                    'in_common':        in_common,
                    'detection_rate_%': rate,
                })

    return pd.DataFrame(records)


# ─── STEP 4: LaTeX tables (one per model) ────────────────────────────────────

def print_latex(table: pd.DataFrame, all_cats: list[str]):
    # Only show categories that have at least one BUMP instance
    active_cats = [c for c in all_cats
                   if table[table['error_category'] == c]['bump_total'].iloc[0] > 0]

    for model in MODEL_ORDER:
        model_df = table[table['model'] == model]

        print(f"\n% ── Model: {model} ──────────────────────────────────────────────")
        print(r"\begin{table}[htbp]")
        print(r"\centering")
        print(f"\\caption{{Error Type Coverage for \\textbf{{{model}}}}}")
        print(f"\\label{{tab:coverage_{model.replace('-', '_').replace(' ', '_')}}}")
        print(r"\resizebox{\columnwidth}{!}{%")
        print(r"\begin{tabular}{l r | r r r | r r r | r r r}")
        print(r"\toprule")
        print(r"& & \multicolumn{3}{c|}{\textit{Minimal}} "
              r"& \multicolumn{3}{c|}{\textit{Method}} "
              r"& \multicolumn{3}{c}{\textit{Class}} \\")
        print(r"\cmidrule(lr){3-5}\cmidrule(lr){6-8}\cmidrule(lr){9-11}")
        print(r"\textbf{Error Category} & \textbf{BUMP $n$} "
              r"& LLM & Com & \% "
              r"& LLM & Com & \% "
              r"& LLM & Com & \% \\")
        print(r"\midrule")

        for cat in active_cats:
            bump_n = model_df[model_df['error_category'] == cat]['bump_total'].iloc[0]
            cells = []
            for variant in VARIANT_ORDER:
                r = model_df[
                    (model_df['context_variant'] == variant) &
                    (model_df['error_category'] == cat)
                ]
                llm  = r['llm_detected'].values[0]
                com  = r['in_common'].values[0]
                rate = r['detection_rate_%'].values[0]
                cells.append(f"{llm} & {com} & {rate}")

            # Escape underscores in category names for LaTeX
            cat_latex = cat.replace('_', r'\_')
            print(f"\\texttt{{{cat_latex}}} & {bump_n} & " + " & ".join(cells) + r" \\")

        print(r"\bottomrule")
        print(r"\end{tabular}%")
        print(r"}")
        print(r"\end{table}")


# ─── MAIN ─────────────────────────────────────────────────────────────────────

def main():
    print("\nStep 1: Loading detected CSV...")
    df = load_detected(DETECTED_CSV)

    print("\nStep 2: Loading BUMP CSV (bump_total from full CSV, rest filtered to detected)...")
    detected_ids = set(df['custom_id'].unique())
    bump_cats, all_cats, bump_total_by_cat = load_bump(BUMP_CSV, detected_ids=detected_ids)

    # Collect any additional error types found by LLM but not in BUMP
    llm_only_cats = set()
    for cats in df['_llm_cats']:
        llm_only_cats.update(cats)
    llm_only_cats -= set(all_cats)  # remove ones already in BUMP
    if llm_only_cats:
        # Append LLM-only categories sorted alphabetically after BUMP categories
        llm_only_sorted = sorted(llm_only_cats)
        all_cats = all_cats + llm_only_sorted
        print(f"\n  Additional LLM-only error types ({len(llm_only_sorted)}):")
        for c in llm_only_sorted:
            print(f"    {c:50s}  (not in BUMP)")

    print("\nStep 3: Building coverage table...")
    table = build_table(bump_total_by_cat, bump_cats, all_cats, df)

    Path(OUTPUT_TABLE_CSV).parent.mkdir(parents=True, exist_ok=True)
    table.to_csv(OUTPUT_TABLE_CSV, index=False)
    print(f"  Table saved → {OUTPUT_TABLE_CSV}")

    print("\nSummary (averaged across model-variants):")
    summary = (
        table[table['bump_total'] > 0]
        .groupby('error_category')
        .agg(
            bump_total=('bump_total', 'first'),
            avg_llm_detected=('llm_detected', 'mean'),
            avg_in_common=('in_common', 'mean'),
            avg_rate=('detection_rate_%', 'mean')
        )
        .round(1)
        .sort_values('avg_rate', ascending=False)
    )
    print(summary.to_string())

    print("\nStep 4: Printing LaTeX tables (one per model)...")
    print_latex(table, all_cats)

    print("\nDone.")


if __name__ == "__main__":
    main()


Step 1: Loading detected CSV...
  Detected CSV: 137 rows | 32 unique instances | 3 models | 3 variants

Step 2: Loading BUMP CSV (bump_total from full CSV, rest filtered to detected)...
  BUMP CSV total: 89 instances
  All unique categories (56):
    MojoFailureException                                full=88  detected=32
    NoClassDefFoundError                                full=35  detected=20
    ClassNotFoundException                              full=30  detected=16
    SocketTimeoutException                              full=15  detected=0
    ClassCastException                                  full=10  detected=7
    AssertionError                                      full=10  detected=0
    RuntimeException                                    full=8  detected=3
    ExceptionInInitializerError                         full=8  detected=1
    UnsupportedClassVersionError                        full=8  detected=0
    BeanInstantiationException                          full=8  dete

In [1]:
"""
analysis1_error_type_coverage_v2.py
====================================
Pipeline:
  1. BUMP CSV      → reads normalized_error_category column
                     all_cats   = ALL unique categories across the full BUMP CSV
                     bump_total = # instances per category from the FULL BUMP CSV
  2. Detected CSV  → per (model, context_variant, instance):
                       bump_bc_errors     : what BUMP ground truth has for this instance
                       llm_detected_errors: what LLM threw for this instance
  3. For each (model, context_variant, error_category):
       bump_total   = # BUMP instances (full CSV) with this category
       llm_detected = # instances where LLM detected this category
       in_common    = # instances where both BUMP and LLM share this category
       rate         = in_common / bump_total * 100

No exclusions — all error types are kept, consistent with v2 pipeline.
"""

import pandas as pd
from collections import defaultdict
from pathlib import Path


# ─── CONFIG ───────────────────────────────────────────────────────────────────
BUMP_CSV         = "/Volumes/Rachna-HD/ConfigFiles/Candidate_BUMP_Instance_errorTypes_with_categories_v2.csv"
DETECTED_CSV     = "/Volumes/Rachna-HD/RQResultsForPaper/RQ3/detected_bc_errortype_coverage_v2.csv"
OUTPUT_TABLE_CSV = "/Volumes/Rachna-HD/RQResultsForPaper/RQ3/analysis1_error_type_coverage_table_v3.csv"

MODEL_ORDER   = ['GPT-4o', 'Qwen-480B', 'GPT-OSS-120b']
VARIANT_ORDER = ['Minimal', 'Method', 'Class']
# ──────────────────────────────────────────────────────────────────────────────


def parse_pipe(val) -> set[str]:
    """Pipe-separated string → set of short class names. No exclusions."""
    if not val or str(val).strip() in ('', 'nan'):
        return set()
    return {e.strip().split('.')[-1] for e in str(val).split('|') if e.strip()}


# ─── STEP 1: Load BUMP CSV ────────────────────────────────────────────────────

def load_bump(bump_csv: str, detected_ids: set[str]) -> tuple[dict[str, set[str]], list[str], dict[str, int]]:
    """
    Reads normalized_error_category from the full BUMP CSV.

    Returns:
      bump_cats         : dict[custom_id → set of categories], restricted to detected_ids
      all_cats          : list of all unique categories, ordered by frequency descending
      bump_total_by_cat : dict[category → count], from ALL rows in BUMP CSV
    """
    df = pd.read_csv(bump_csv, sep=None, engine='python', dtype=str, keep_default_na=False)
    df['custom_id'] = df['custom_id'].astype(str).str.strip()

    bump_cats: dict[str, set[str]] = {}
    all_counts: dict[str, int] = defaultdict(int)
    detected_counts: dict[str, int] = defaultdict(int)

    for _, row in df.iterrows():
        cid  = row['custom_id']
        cats = parse_pipe(row.get('normalized_error_category', ''))
        # bump_total always from full BUMP CSV
        for c in cats:
            all_counts[c] += 1
        # bump_cats restricted to detected_ids (for in_common calculation)
        if cid in detected_ids:
            bump_cats[cid] = cats
            for c in cats:
                detected_counts[c] += 1

    # Order by frequency descending (full BUMP)
    all_cats = sorted(all_counts.keys(), key=lambda c: -all_counts[c])

    print(f"  BUMP CSV total: {len(df)} instances")
    print(f"  All unique categories ({len(all_cats)}):")
    for c in all_cats:
        print(f"    {c:50s}  full={all_counts[c]}  detected={detected_counts.get(c, 0)}")

    return bump_cats, all_cats, dict(all_counts)


# ─── STEP 2: Load detected CSV ────────────────────────────────────────────────

def load_detected(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, sep=None, engine='python', dtype=str, keep_default_na=False)

    df['custom_id'] = df['custom_id'].astype(str).str.strip()

    # Parse bump_bc_errors and llm_detected_errors into sets of short class names
    df['_bump_cats'] = df['bump_bc_errors'].apply(parse_pipe)
    df['_llm_cats']  = df['llm_detected_errors'].apply(parse_pipe)

    print(f"  Detected CSV: {len(df)} rows | "
          f"{df['custom_id'].nunique()} unique instances | "
          f"{df['model'].nunique()} models | "
          f"{df['context_variant'].nunique()} variants")
    return df


# ─── STEP 3: Build coverage table ────────────────────────────────────────────

def build_table(
    bump_total_by_cat: dict[str, int],
    bump_cats: dict[str, set[str]],
    all_cats: list[str],
    df: pd.DataFrame,
) -> pd.DataFrame:
    """
    For each (model, context_variant, error_category):
      bump_total   : from full BUMP CSV (how many BUMP instances have this error type)
      llm_detected : # instances in detected CSV where LLM threw this error type
      in_common    : # instances where BUMP has this error type AND LLM also detected it
      rate         : in_common / bump_total * 100
    """
    records = []
    for model in MODEL_ORDER:
        for variant in VARIANT_ORDER:
            sub = df[(df['model'] == model) & (df['context_variant'] == variant)]

            for cat in all_cats:
                bump_total = bump_total_by_cat.get(cat, 0)

                # llm_detected: count instances where LLM detected this error type
                llm_detected = int(sub['_llm_cats'].apply(lambda s: cat in s).sum())

                # in_common: count instances where BUMP has this error type
                # AND LLM also detected the same error type
                in_common = int(sub.apply(
                    lambda row: cat in bump_cats.get(row['custom_id'], set())
                                and cat in row['_llm_cats'],
                    axis=1
                ).sum())

                rate = round(in_common / bump_total * 100, 1) if bump_total > 0 else 0.0

                records.append({
                    'model':            model,
                    'context_variant':  variant,
                    'error_category':   cat,
                    'bump_total':       bump_total,
                    'llm_detected':     llm_detected,
                    'in_common':        in_common,
                    'detection_rate_%': rate,
                })

    return pd.DataFrame(records)


# ─── STEP 4: LaTeX tables (one per model) ────────────────────────────────────

def print_latex(table: pd.DataFrame, all_cats: list[str]):
    # Only show categories that have at least one BUMP instance
    active_cats = [c for c in all_cats
                   if table[table['error_category'] == c]['bump_total'].iloc[0] > 0]

    for model in MODEL_ORDER:
        model_df = table[table['model'] == model]

        print(f"\n% ── Model: {model} ──────────────────────────────────────────────")
        print(r"\begin{table}[htbp]")
        print(r"\centering")
        print(f"\\caption{{Error Type Coverage for \\textbf{{{model}}}}}")
        print(f"\\label{{tab:coverage_{model.replace('-', '_').replace(' ', '_')}}}")
        print(r"\resizebox{\columnwidth}{!}{%")
        print(r"\begin{tabular}{l r | r r r | r r r | r r r}")
        print(r"\toprule")
        print(r"& & \multicolumn{3}{c|}{\textit{Minimal}} "
              r"& \multicolumn{3}{c|}{\textit{Method}} "
              r"& \multicolumn{3}{c}{\textit{Class}} \\")
        print(r"\cmidrule(lr){3-5}\cmidrule(lr){6-8}\cmidrule(lr){9-11}")
        print(r"\textbf{Error Category} & \textbf{BUMP $n$} "
              r"& LLM & Com & \% "
              r"& LLM & Com & \% "
              r"& LLM & Com & \% \\")
        print(r"\midrule")

        for cat in active_cats:
            bump_n = model_df[model_df['error_category'] == cat]['bump_total'].iloc[0]
            cells = []
            for variant in VARIANT_ORDER:
                r = model_df[
                    (model_df['context_variant'] == variant) &
                    (model_df['error_category'] == cat)
                ]
                llm  = r['llm_detected'].values[0]
                com  = r['in_common'].values[0]
                rate = r['detection_rate_%'].values[0]
                cells.append(f"{llm} & {com} & {rate}")

            # Escape underscores in category names for LaTeX
            cat_latex = cat.replace('_', r'\_')
            print(f"\\texttt{{{cat_latex}}} & {bump_n} & " + " & ".join(cells) + r" \\")

        print(r"\bottomrule")
        print(r"\end{tabular}%")
        print(r"}")
        print(r"\end{table}")


# ─── MAIN ─────────────────────────────────────────────────────────────────────

def main():
    print("\nStep 1: Loading detected CSV...")
    df = load_detected(DETECTED_CSV)

    print("\nStep 2: Loading BUMP CSV (bump_total from full CSV, rest filtered to detected)...")
    detected_ids = set(df['custom_id'].unique())
    bump_cats, all_cats, bump_total_by_cat = load_bump(BUMP_CSV, detected_ids=detected_ids)

    # Collect any additional error types found by LLM but not in BUMP
    llm_only_cats = set()
    for cats in df['_llm_cats']:
        llm_only_cats.update(cats)
    llm_only_cats -= set(all_cats)  # remove ones already in BUMP
    if llm_only_cats:
        # Append LLM-only categories sorted alphabetically after BUMP categories
        llm_only_sorted = sorted(llm_only_cats)
        all_cats = all_cats + llm_only_sorted
        print(f"\n  Additional LLM-only error types ({len(llm_only_sorted)}):")
        for c in llm_only_sorted:
            print(f"    {c:50s}  (not in BUMP)")

    print("\nStep 3: Building coverage table...")
    table = build_table(bump_total_by_cat, bump_cats, all_cats, df)

    Path(OUTPUT_TABLE_CSV).parent.mkdir(parents=True, exist_ok=True)
    table.to_csv(OUTPUT_TABLE_CSV, index=False)
    print(f"  Table saved → {OUTPUT_TABLE_CSV}")

    print("\nSummary (averaged across model-variants):")
    summary = (
        table[table['bump_total'] > 0]
        .groupby('error_category')
        .agg(
            bump_total=('bump_total', 'first'),
            avg_llm_detected=('llm_detected', 'mean'),
            avg_in_common=('in_common', 'mean'),
            avg_rate=('detection_rate_%', 'mean')
        )
        .round(1)
        .sort_values('avg_rate', ascending=False)
    )
    print(summary.to_string())

    print("\nStep 4: Printing LaTeX tables (one per model)...")
    print_latex(table, all_cats)

    print("\nDone.")


if __name__ == "__main__":
    main()


Step 1: Loading detected CSV...
  Detected CSV: 137 rows | 32 unique instances | 3 models | 3 variants

Step 2: Loading BUMP CSV (bump_total from full CSV, rest filtered to detected)...
  BUMP CSV total: 89 instances
  All unique categories (56):
    MojoFailureException                                full=88  detected=32
    NoClassDefFoundError                                full=35  detected=20
    ClassNotFoundException                              full=30  detected=16
    SocketTimeoutException                              full=15  detected=0
    ClassCastException                                  full=10  detected=7
    AssertionError                                      full=10  detected=0
    RuntimeException                                    full=8  detected=3
    ExceptionInInitializerError                         full=8  detected=1
    UnsupportedClassVersionError                        full=8  detected=0
    BeanInstantiationException                          full=8  dete

In [2]:
"""
analysis2_detection_characterization.py
========================================
RQ3 Part 2: How do LLM-generated tests detect breaking changes?

For each (model, context_variant), classifies every detected instance into:

  1. Detection Match Type (per instance):
     - EXACT_MATCH    : all BUMP errors found by LLM (missed=0, matched>0)
     - PARTIAL_MATCH  : some BUMP errors found, some missed (matched>0, missed>0)
     - DIFFERENT_ERROR: LLM detected the BC (test failed) but with entirely
                        different error types than BUMP (matched=0, new>0)
     - NO_BUMP_ERRORS : BUMP has no error types for this instance but LLM
                        detected failures (bump_total=0, llm_total>0)

  2. Failure Category Analysis:
     - Detection rate per failureCategory (from BUMP CSV)
     - Which failure categories are easier/harder to detect

  3. Error Path Analysis:
     - How often LLM finds additional errors beyond BUMP ground truth
     - Distribution of matched vs new vs missed per instance

Reads from the detected CSV (output of detected_bc_errortype_coverage_v2.py).
"""

import pandas as pd
from collections import defaultdict
from pathlib import Path


# ─── CONFIG ───────────────────────────────────────────────────────────────────
DETECTED_CSV     = "/Volumes/Rachna-HD/RQResultsForPaper/RQ3/detected_bc_errortype_coverage_v2.csv"
BUMP_CSV         = "/Volumes/Rachna-HD/ConfigFiles/Candidate_BUMP_Instance_errorTypes_with_categories_v2.csv"
OUTPUT_DIR       = "/Volumes/Rachna-HD/RQResultsForPaper/RQ3/"

MODEL_ORDER   = ['GPT-4o', 'Qwen-480B', 'GPT-OSS-120b']
VARIANT_ORDER = ['Minimal', 'Method', 'Class']
# ──────────────────────────────────────────────────────────────────────────────


def parse_pipe(val) -> set:
    if not val or str(val).strip() in ('', 'nan'):
        return set()
    return {e.strip() for e in str(val).split('|') if e.strip()}


def classify_instance(row) -> str:
    """Classify a single detected instance by its match type."""
    matched = int(row['matched_count'])
    missed  = int(row['missed_count'])
    new     = int(row['new_count'])
    bump_t  = int(row['bump_total'])
    llm_t   = int(row['llm_total'])

    if bump_t == 0 and llm_t > 0:
        return 'NO_BUMP_ERRORS'
    elif matched > 0 and missed == 0:
        return 'EXACT_MATCH'
    elif matched > 0 and missed > 0:
        return 'PARTIAL_MATCH'
    elif matched == 0 and new > 0:
        return 'DIFFERENT_ERROR'
    else:
        return 'UNKNOWN'


def load_detected() -> pd.DataFrame:
    df = pd.read_csv(DETECTED_CSV, sep=None, engine='python', dtype=str, keep_default_na=False)
    # Convert numeric columns
    for col in ['bump_total', 'llm_total', 'matched_count', 'missed_count', 'new_count',
                'match_rate_%', 'num_failed_tests']:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
    df['custom_id'] = df['custom_id'].astype(str).str.strip()
    return df


def load_bump_failure_categories() -> dict:
    """Load failureCategory for all BUMP instances."""
    df = pd.read_csv(BUMP_CSV, sep=None, engine='python', dtype=str, keep_default_na=False)
    df['custom_id'] = df['custom_id'].astype(str).str.strip()
    return dict(zip(df['custom_id'], df.get('failureCategory', '')))


# ─── ANALYSIS 1: Detection Match Type Classification ─────────────────────────

def analysis_match_types(df: pd.DataFrame):
    """Classify each instance and summarize per (model, variant)."""
    df = df.copy()
    df['match_type'] = df.apply(classify_instance, axis=1)

    print("\n" + "=" * 80)
    print("ANALYSIS 1: Detection Match Type Classification")
    print("=" * 80)

    # Summary table
    records = []
    for model in MODEL_ORDER:
        for variant in VARIANT_ORDER:
            sub = df[(df['model'] == model) & (df['context_variant'] == variant)]
            total = len(sub)
            if total == 0:
                continue

            counts = sub['match_type'].value_counts().to_dict()
            exact    = counts.get('EXACT_MATCH', 0)
            partial  = counts.get('PARTIAL_MATCH', 0)
            diff     = counts.get('DIFFERENT_ERROR', 0)
            no_bump  = counts.get('NO_BUMP_ERRORS', 0)
            unknown  = counts.get('UNKNOWN', 0)

            records.append({
                'model': model,
                'context_variant': variant,
                'total_detected': total,
                'exact_match': exact,
                'exact_match_%': round(exact / total * 100, 1),
                'partial_match': partial,
                'partial_match_%': round(partial / total * 100, 1),
                'different_error': diff,
                'different_error_%': round(diff / total * 100, 1),
                'no_bump_errors': no_bump,
                'no_bump_errors_%': round(no_bump / total * 100, 1),
                'unknown': unknown,
            })

            print(f"\n  {model} | {variant} ({total} detected instances):")
            print(f"    EXACT_MATCH    : {exact:3d} ({exact/total*100:5.1f}%) — all BUMP errors reproduced")
            print(f"    PARTIAL_MATCH  : {partial:3d} ({partial/total*100:5.1f}%) — some BUMP errors reproduced")
            print(f"    DIFFERENT_ERROR: {diff:3d} ({diff/total*100:5.1f}%) — BC detected via different errors")
            print(f"    NO_BUMP_ERRORS : {no_bump:3d} ({no_bump/total*100:5.1f}%) — BUMP has no error types listed")
            if unknown > 0:
                print(f"    UNKNOWN        : {unknown:3d}")

    result_df = pd.DataFrame(records)
    out_path = Path(OUTPUT_DIR) / "analysis2_match_types.csv"
    result_df.to_csv(out_path, index=False)
    print(f"\n  Saved → {out_path}")

    return df  # return with match_type column added


# ─── ANALYSIS 2: Failure Category Detection Rates ────────────────────────────

def analysis_failure_categories(df: pd.DataFrame, bump_fc: dict):
    """Detection rate per BUMP failureCategory."""
    print("\n" + "=" * 80)
    print("ANALYSIS 2: Detection Rate by BUMP Failure Category")
    print("=" * 80)

    # Count total BUMP instances per failureCategory
    fc_total = defaultdict(int)
    for cid, fc in bump_fc.items():
        if fc:
            fc_total[fc] += 1

    print(f"\n  BUMP failure categories (total instances):")
    for fc, cnt in sorted(fc_total.items(), key=lambda x: -x[1]):
        print(f"    {fc:40s}  {cnt}")

    records = []
    for model in MODEL_ORDER:
        for variant in VARIANT_ORDER:
            sub = df[(df['model'] == model) & (df['context_variant'] == variant)]

            # Count detected instances per failureCategory
            fc_detected = defaultdict(int)
            fc_exact = defaultdict(int)
            fc_partial = defaultdict(int)
            fc_diff = defaultdict(int)

            for _, row in sub.iterrows():
                fc = row.get('bump_bc_failure_category', '')
                if not fc:
                    fc = bump_fc.get(row['custom_id'], '')
                if not fc:
                    continue
                fc_detected[fc] += 1
                mt = row.get('match_type', '')
                if mt == 'EXACT_MATCH':
                    fc_exact[fc] += 1
                elif mt == 'PARTIAL_MATCH':
                    fc_partial[fc] += 1
                elif mt == 'DIFFERENT_ERROR':
                    fc_diff[fc] += 1

            for fc in sorted(fc_total.keys()):
                total = fc_total[fc]
                detected = fc_detected.get(fc, 0)
                rate = round(detected / total * 100, 1) if total > 0 else 0.0

                records.append({
                    'model': model,
                    'context_variant': variant,
                    'failure_category': fc,
                    'bump_total': total,
                    'detected': detected,
                    'detection_rate_%': rate,
                    'exact_match': fc_exact.get(fc, 0),
                    'partial_match': fc_partial.get(fc, 0),
                    'different_error': fc_diff.get(fc, 0),
                })

    result_df = pd.DataFrame(records)
    out_path = Path(OUTPUT_DIR) / "analysis2_failure_categories.csv"
    result_df.to_csv(out_path, index=False)
    print(f"\n  Saved → {out_path}")

    # Print summary per failure category (averaged across model-variants)
    print("\n  Summary (averaged across all model-variants):")
    summary = (
        result_df[result_df['bump_total'] > 0]
        .groupby('failure_category')
        .agg(
            bump_total=('bump_total', 'first'),
            avg_detected=('detected', 'mean'),
            avg_rate=('detection_rate_%', 'mean'),
            avg_exact=('exact_match', 'mean'),
            avg_partial=('partial_match', 'mean'),
            avg_diff=('different_error', 'mean'),
        )
        .round(1)
        .sort_values('avg_rate', ascending=False)
    )
    print(summary.to_string())

    return result_df


# ─── ANALYSIS 3: Error Path Analysis ─────────────────────────────────────────

def analysis_error_paths(df: pd.DataFrame):
    """Analyze how LLM tests find errors — same path vs different path."""
    print("\n" + "=" * 80)
    print("ANALYSIS 3: Error Path Analysis")
    print("=" * 80)

    records = []
    for model in MODEL_ORDER:
        for variant in VARIANT_ORDER:
            sub = df[(df['model'] == model) & (df['context_variant'] == variant)]
            total = len(sub)
            if total == 0:
                continue

            # Instances where LLM found additional errors beyond BUMP
            has_new = sub[sub['new_count'] > 0]
            # Instances where LLM found only BUMP errors (no new ones)
            only_matched = sub[(sub['matched_count'] > 0) & (sub['new_count'] == 0)]
            # Instances with both matched and new
            both = sub[(sub['matched_count'] > 0) & (sub['new_count'] > 0)]

            # Average counts per instance
            avg_matched = sub['matched_count'].mean()
            avg_missed  = sub['missed_count'].mean()
            avg_new     = sub['new_count'].mean()
            avg_bump_t  = sub['bump_total'].mean()
            avg_llm_t   = sub['llm_total'].mean()

            records.append({
                'model': model,
                'context_variant': variant,
                'total_detected': total,
                'instances_with_new_errors': len(has_new),
                'instances_with_new_errors_%': round(len(has_new) / total * 100, 1),
                'instances_only_matched': len(only_matched),
                'instances_only_matched_%': round(len(only_matched) / total * 100, 1),
                'instances_matched_and_new': len(both),
                'instances_matched_and_new_%': round(len(both) / total * 100, 1),
                'avg_matched_per_instance': round(avg_matched, 2),
                'avg_missed_per_instance': round(avg_missed, 2),
                'avg_new_per_instance': round(avg_new, 2),
                'avg_bump_errors_per_instance': round(avg_bump_t, 2),
                'avg_llm_errors_per_instance': round(avg_llm_t, 2),
            })

            print(f"\n  {model} | {variant} ({total} instances):")
            print(f"    Avg BUMP error types per instance : {avg_bump_t:.2f}")
            print(f"    Avg LLM error types per instance  : {avg_llm_t:.2f}")
            print(f"    Avg matched per instance           : {avg_matched:.2f}")
            print(f"    Avg missed per instance            : {avg_missed:.2f}")
            print(f"    Avg new (LLM-only) per instance    : {avg_new:.2f}")
            print(f"    Instances with new errors beyond BUMP: {len(has_new):3d} ({len(has_new)/total*100:.1f}%)")
            print(f"    Instances with only matched errors   : {len(only_matched):3d} ({len(only_matched)/total*100:.1f}%)")
            print(f"    Instances with matched + new errors  : {len(both):3d} ({len(both)/total*100:.1f}%)")

    result_df = pd.DataFrame(records)
    out_path = Path(OUTPUT_DIR) / "analysis2_error_paths.csv"
    result_df.to_csv(out_path, index=False)
    print(f"\n  Saved → {out_path}")

    return result_df


# ─── ANALYSIS 4: LaTeX Tables ────────────────────────────────────────────────

def print_latex_match_types(df: pd.DataFrame):
    """LaTeX table for match type classification."""
    print("\n% ── Match Type Classification Table ──")
    print(r"\begin{table}[t]")
    print(r"\centering")
    print(r"\caption{Detection Match Type Classification per Model and Context Variant.}")
    print(r"\label{tab:match_types}")
    print(r"\scriptsize")
    print(r"\begin{tabular}{ll r rr rr rr rr}")
    print(r"\toprule")
    print(r"\textbf{Model} & \textbf{Variant} & \textbf{Det.}"
          r" & \multicolumn{2}{c}{\textbf{Exact}} & \multicolumn{2}{c}{\textbf{Partial}}"
          r" & \multicolumn{2}{c}{\textbf{Different}} & \multicolumn{2}{c}{\textbf{No BUMP}} \\")
    print(r"\cmidrule(lr){4-5} \cmidrule(lr){6-7} \cmidrule(lr){8-9} \cmidrule(lr){10-11}")
    print(r" & & & $n$ & \% & $n$ & \% & $n$ & \% & $n$ & \% \\")
    print(r"\midrule")

    for model in MODEL_ORDER:
        sub = df[(df['model'] == model)]
        first = True
        for variant in VARIANT_ORDER:
            row = sub[sub['context_variant'] == variant]
            if row.empty:
                continue
            r = row.iloc[0]
            total = int(r['total_detected'])
            exact = int(r['exact_match'])
            partial = int(r['partial_match'])
            diff = int(r['different_error'])
            no_bump = int(r['no_bump_errors'])

            model_col = f"\\textbf{{{model}}}" if first else ""
            first = False

            print(f"{model_col} & {variant} & {total}"
                  f" & {exact} & {exact/total*100:.1f}"
                  f" & {partial} & {partial/total*100:.1f}"
                  f" & {diff} & {diff/total*100:.1f}"
                  f" & {no_bump} & {no_bump/total*100:.1f}"
                  r" \\")
        if model != MODEL_ORDER[-1]:
            print(r"\midrule")

    print(r"\bottomrule")
    print(r"\end{tabular}")
    print(r"\end{table}")


def print_latex_failure_categories(fc_df: pd.DataFrame):
    """LaTeX table for failure category detection rates."""
    fc_list = sorted(
        fc_df['failure_category'].unique(),
        key=lambda fc: -fc_df[fc_df['failure_category'] == fc]['bump_total'].iloc[0]
    )

    print("\n% ── Failure Category Detection Table ──")
    print(r"\begin{table*}[t]")
    print(r"\centering")
    print(r"\caption{Detection Rate by BUMP Failure Category.}")
    print(r"\label{tab:failure_cat_detection}")
    print(r"\scriptsize")
    print(r"\setlength{\tabcolsep}{2.5pt}")
    print(r"\begin{tabular}{l r | rrr rrr rrr | rrr rrr rrr | rrr rrr rrr}")
    print(r"\toprule")
    print(r"& & \multicolumn{9}{c|}{\textbf{GPT-4o}}"
          r" & \multicolumn{9}{c|}{\textbf{Qwen-480B}}"
          r" & \multicolumn{9}{c}{\textbf{GPT-OSS-120b}} \\")
    print(r"\cmidrule(lr){3-11} \cmidrule(lr){12-20} \cmidrule(lr){21-29}")
    print(r"& & \multicolumn{3}{c}{\textit{Min.}} & \multicolumn{3}{c}{\textit{Meth.}} & \multicolumn{3}{c|}{\textit{Class}}"
          r" & \multicolumn{3}{c}{\textit{Min.}} & \multicolumn{3}{c}{\textit{Meth.}} & \multicolumn{3}{c|}{\textit{Class}}"
          r" & \multicolumn{3}{c}{\textit{Min.}} & \multicolumn{3}{c}{\textit{Meth.}} & \multicolumn{3}{c}{\textit{Class}} \\")
    print(r"\cmidrule(lr){3-5} \cmidrule(lr){6-8} \cmidrule(lr){9-11}"
          r" \cmidrule(lr){12-14} \cmidrule(lr){15-17} \cmidrule(lr){18-20}"
          r" \cmidrule(lr){21-23} \cmidrule(lr){24-26} \cmidrule(lr){27-29}")
    print(r"\textbf{Failure Category} & \textbf{$n$}"
          r" & D & E & \% & D & E & \% & D & E & \%"
          r" & D & E & \% & D & E & \% & D & E & \%"
          r" & D & E & \% & D & E & \% & D & E & \% \\")
    print(r"\midrule")

    for fc in fc_list:
        fc_rows = fc_df[fc_df['failure_category'] == fc]
        bump_n = int(fc_rows['bump_total'].iloc[0])
        if bump_n == 0:
            continue

        cells = []
        for model in MODEL_ORDER:
            for variant in VARIANT_ORDER:
                r = fc_rows[
                    (fc_rows['model'] == model) &
                    (fc_rows['context_variant'] == variant)
                ]
                if len(r) == 0:
                    cells.append("0 & 0 & --")
                else:
                    det = int(r['detected'].values[0])
                    exact = int(r['exact_match'].values[0])
                    rate = float(r['detection_rate_%'].values[0])
                    rate_str = '--' if rate == 0 else f'{rate:g}'
                    cells.append(f"{det} & {exact} & {rate_str}")

        fc_latex = fc.replace('_', r'\_')
        print(f"\\texttt{{{fc_latex}}} & {bump_n} & " + " & ".join(cells) + r" \\")

    print(r"\bottomrule")
    print(r"\end{tabular}")
    print(r"\end{table*}")


# ─── MAIN ─────────────────────────────────────────────────────────────────────

def main():
    Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

    print("Loading data...")
    df = load_detected()
    bump_fc = load_bump_failure_categories()
    print(f"  Detected CSV: {len(df)} rows")
    print(f"  BUMP failure categories: {len(bump_fc)} instances")

    # Analysis 1: Match type classification
    df = analysis_match_types(df)

    # Analysis 2: Failure category detection rates
    fc_df = analysis_failure_categories(df, bump_fc)

    # Analysis 3: Error path analysis
    analysis_error_paths(df)

    # Analysis 4: LaTeX tables
    print("\n" + "=" * 80)
    print("LaTeX TABLES")
    print("=" * 80)

    # Build summary df for match types LaTeX
    match_records = []
    for model in MODEL_ORDER:
        for variant in VARIANT_ORDER:
            sub = df[(df['model'] == model) & (df['context_variant'] == variant)]
            total = len(sub)
            if total == 0:
                continue
            counts = sub['match_type'].value_counts().to_dict()
            match_records.append({
                'model': model,
                'context_variant': variant,
                'total_detected': total,
                'exact_match': counts.get('EXACT_MATCH', 0),
                'partial_match': counts.get('PARTIAL_MATCH', 0),
                'different_error': counts.get('DIFFERENT_ERROR', 0),
                'no_bump_errors': counts.get('NO_BUMP_ERRORS', 0),
            })
    match_df = pd.DataFrame(match_records)

    print_latex_match_types(match_df)
    print_latex_failure_categories(fc_df)

    print("\n\nDone. All results saved to:", OUTPUT_DIR)


if __name__ == "__main__":
    main()

Loading data...
  Detected CSV: 137 rows
  BUMP failure categories: 89 instances

ANALYSIS 1: Detection Match Type Classification

  GPT-4o | Minimal (17 detected instances):
    EXACT_MATCH    :   6 ( 35.3%) — all BUMP errors reproduced
    PARTIAL_MATCH  :   9 ( 52.9%) — some BUMP errors reproduced
    DIFFERENT_ERROR:   0 (  0.0%) — BC detected via different errors
    NO_BUMP_ERRORS :   0 (  0.0%) — BUMP has no error types listed
    UNKNOWN        :   2

  GPT-4o | Method (13 detected instances):
    EXACT_MATCH    :   9 ( 69.2%) — all BUMP errors reproduced
    PARTIAL_MATCH  :   2 ( 15.4%) — some BUMP errors reproduced
    DIFFERENT_ERROR:   0 (  0.0%) — BC detected via different errors
    NO_BUMP_ERRORS :   0 (  0.0%) — BUMP has no error types listed
    UNKNOWN        :   2

  GPT-4o | Class (27 detected instances):
    EXACT_MATCH    :  11 ( 40.7%) — all BUMP errors reproduced
    PARTIAL_MATCH  :  12 ( 44.4%) — some BUMP errors reproduced
    DIFFERENT_ERROR:   0 (  0.0%) —